# TMDB Movie Data Analysis using Pandas and APIs
**Step 1: API Data Extraction & Step 2: Data Cleaning & Transformation**

This notebook fetches movie data from the TMDb API, cleans it, and prepares it for KPI analysis (Step 3) and visualization (Step 4).

## Step 0: Load API credentials
API key is stored in a `.env` file at the project root.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")
api_key = os.getenv("TMDB_API_KEY")
print("Key loaded:", api_key is not None)


## Step 1: Fetch Movie Data from API
First, a single test call to inspect the raw JSON shape before looping over all movie IDs.

In [ ]:
import requests

MOVIE_URL = os.getenv("URL")
api_key = os.getenv("TMDB_API_KEY")

movie_id = 597  # Titanic


def fetch_movie(movie_id):
    url = f"{MOVIE_URL}{movie_id}"

    params = {
        "api_key": api_key
    }

    response = requests.get(
        url,
        params=params,
        timeout=15
    )

    # Raise an exception for HTTP errors
    response.raise_for_status()

    return response.json()


data = fetch_movie(movie_id)
data


### Fetch all 19 movie IDs from the project brief


In [ ]:
import json
from pathlib import Path

# 2. Movie IDs required by the project

MOVIE_IDS = [
    0, 299534, 19995, 140607, 299536, 597, 135397,
    420818, 24428, 168259, 99861, 284054, 12445,
    181808, 330457, 351286, 109445, 321612, 260513
]


# 3. File where the raw dataset will be stored

OUTPUT_DIR = Path("../data/raw")
OUTPUT_FILE = OUTPUT_DIR / "movies.json"


# 4. Check whether a valid dataset already exists

def dataset_exists_and_is_valid():
    """
    Return True if movies.json exists and contains
    a non-empty list of movie records.
    """

    if not OUTPUT_FILE.exists():
        return False

    try:
        with open(OUTPUT_FILE, "r", encoding="utf-8") as file:
            data = json.load(file)

        if not isinstance(data, list):
            return False

        if len(data) == 0:
            return False

        return True

    except (json.JSONDecodeError, OSError):
        return False


# 5. Fetch all required movies

def download_movies():

    print("Downloading movie data from TMDB...")

    movies = []

    for movie_id in MOVIE_IDS:

        print(f"Fetching movie ID: {movie_id}")

        try:
            movie = fetch_movie(movie_id)

            # Make sure the response actually contains data
            if movie and isinstance(movie, dict):

                # TMDB error responses can contain an "status_code"
                if "status_code" in movie:
                    print(
                        f"  Skipping ID {movie_id}: "
                        f"{movie.get('status_message', 'API error')}"
                    )
                    continue

                movies.append(movie)

            else:
                print(f"  Skipping ID {movie_id}: empty response.")

        except requests.exceptions.Timeout:
            print(f"  Timeout while fetching ID {movie_id}.")

        except requests.exceptions.ConnectionError:
            print(
                f"  Could not connect to TMDB while fetching "
                f"ID {movie_id}."
            )

        except requests.exceptions.HTTPError as error:
            print(
                f"  HTTP error for ID {movie_id}: "
                f"{error}"
            )

        except requests.exceptions.RequestException as error:
            print(
                f"  Request failed for ID {movie_id}: "
                f"{error}"
            )

    # Make sure we didn't download an empty dataset

    if not movies:
        raise RuntimeError(
            "No movie data was downloaded. "
            "The dataset will NOT be saved."
        )

    # Create the directory if it doesn't exist

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # Save the raw dataset

    with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
        json.dump(
            movies,
            file,
            indent=2,
            ensure_ascii=False
        )

    print()
    print(f"Successfully downloaded {len(movies)} movies.")
    print(f"Dataset saved to: {OUTPUT_FILE}")


# 6. Run: skip the download if a valid dataset already exists

if dataset_exists_and_is_valid():
    print(f"Dataset already exists at: {OUTPUT_FILE}")
    print("Skipping API download.")
else:
    download_movies()


# 7. Load the raw dataset into memory for the next steps

with open(OUTPUT_FILE, "r", encoding="utf-8") as file:
    movies_data = json.load(file)

print(f"Loaded {len(movies_data)} movies into memory.")


In [ ]:
import pandas as pd

raw_df = pd.DataFrame(movies_data)
raw_df.head()

## Step 2: Data Cleaning and Preprocessing
### Drop irrelevant columns

In [ ]:
columns_to_drop = ['adult', 'imdb_id', 'original_title', 'video', 'homepage']

df = raw_df.drop(columns=columns_to_drop)

print(f"Columns before: {raw_df.shape[1]}, Columns after: {df.shape[1]}")
df.head()


### Investigate and drop the undocumented `softcore` field
Not found in any official TMDb documentation. Behaves like `adult` (boolean content flag, `False` for every movie here) — irrelevant to financial/performance analysis, so it's dropped.

In [ ]:
if 'softcore' in df.columns:
    df = df.drop(columns=['softcore'])
print(f"Columns now: {df.shape[1]}")


### Extract JSON-nested columns into clean, readable strings
`genres`, `production_companies`, `production_countries`, and `spoken_languages` are all lists of dictionaries. Each is reduced to a single `|`-separated string of names.

In [ ]:
def extract_genre_names(genre_list):
    if isinstance(genre_list, list):
        return "|".join([g['name'] for g in genre_list])
    return None

df['genres'] = df['genres'].apply(extract_genre_names)
df['genres'].head()


In [ ]:
def extract_names(item_list):
    if isinstance(item_list, list):
        return "|".join([item['name'] for item in item_list])
    return None

df['production_companies'] = df['production_companies'].apply(extract_names)
df['production_countries'] = df['production_countries'].apply(extract_names)
df['spoken_languages'] = df['spoken_languages'].apply(extract_names)


`belongs_to_collection` is a single dictionary (or `None`), not a list — handled with `.get('name')` so a missing key never raises an error.

In [ ]:
def extract_collection_name(collection):
    if isinstance(collection, dict):
        return collection.get('name')
    return None

df['belongs_to_collection'] = df['belongs_to_collection'].apply(extract_collection_name)


`origin_country` is a plain list of country codes (no nested dictionaries) — joined directly.

In [ ]:
df['origin_country'] = df['origin_country'].apply(lambda c: "|".join(c) if isinstance(c, list) else c)
df['origin_country'].head()

In [ ]:
df[['genres', 'belongs_to_collection', 'production_companies',
    'production_countries', 'spoken_languages']].head()


### Inspect extracted columns with `value_counts()`
Check the five JSON-derived columns for anomalies — empty strings from empty lists, a single dominant value, unexpected one-off labels, etc. — before moving on to dtype conversion.

In [ ]:
extracted_cols = ['genres', 'belongs_to_collection', 'production_countries',
                   'production_companies', 'spoken_languages']

for col in extracted_cols:
    print(f"--- {col} ---")
    print(f"Missing/empty: {df[col].isna().sum() + (df[col] == '').sum()}")
    print(df[col].value_counts().head(10))
    print()


### Convert data types
`errors='coerce'` converts unparseable values to `NaN`/`NaT` instead of crashing.

In [ ]:
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')
df['id'] = pd.to_numeric(df['id'], errors='coerce')
df['popularity'] = pd.to_numeric(df['popularity'], errors='coerce')

df[['budget', 'id', 'popularity']].dtypes


In [ ]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year
df['release_date'].head()

### Replace unrealistic zero values
A budget/revenue/runtime of exactly `0` is not physically plausible for a real released movie — it signals missing data, not a true value of zero.

In [ ]:
import numpy as np

df['budget'] = df['budget'].replace(0, np.nan)
df['revenue'] = df['revenue'].replace(0, np.nan)
df['runtime'] = df['runtime'].replace(0, np.nan)

print("Missing budget:", df['budget'].isna().sum())
print("Missing revenue:", df['revenue'].isna().sum())
print("Missing runtime:", df['runtime'].isna().sum())


### Convert budget & revenue to million USD

In [ ]:
df['budget_musd'] = df['budget'] / 1_000_000
df['revenue_musd'] = df['revenue'] / 1_000_000

df[['title', 'budget_musd', 'revenue_musd']].head()


### Check for `vote_count = 0` anomalies
If `vote_count` is 0, `vote_average` has no real data behind it and should be treated as missing.

In [ ]:
zero_votes = df[df['vote_count'] == 0]
print(f"Movies with 0 vote_count: {len(zero_votes)}")
zero_votes[['title', 'vote_count', 'vote_average']]


In [ ]:
df.loc[df['vote_count'] == 0, 'vote_average'] = np.nan

### Check for placeholder text in `overview` / `tagline`

In [ ]:
print(df['overview'].value_counts().head(10))
print(df['tagline'].value_counts().head(10))

In [ ]:
df['overview'] = df['overview'].replace('No Data', np.nan)
df['tagline'] = df['tagline'].replace('No Data', np.nan)


### Remove duplicates and rows with unknown `id` / `title`

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Missing id:", df['id'].isna().sum())
print("Missing title:", df['title'].isna().sum())

df = df.drop_duplicates()
df = df.dropna(subset=['id', 'title'])

### Keep only rows with at least 10 non-NaN columns

In [ ]:
non_null_counts = df.notna().sum(axis=1)
print(non_null_counts.sort_values())

df = df[non_null_counts >= 10]
print(f"Rows remaining: {len(df)}")


### Filter to 'Released' movies only, then drop `status`

In [ ]:
print(df['status'].value_counts())

In [ ]:
df = df[df['status'] == 'Released']
df = df.drop(columns=['status'])
print(f"Rows remaining: {len(df)}")
print(f"Columns remaining: {df.shape[1]}")


### Fetch cast & crew (credits endpoint)
`cast`, `cast_size`, `director`, and `crew_size` don't exist in the `/movie/{id}` response — they come from a separate endpoint: `/movie/{id}/credits`. One more API call per movie, following the exact same pattern as Step 1.

The response shape: a dictionary with `'cast'` (list of actor dicts, each with a `'name'`) and `'crew'` (list of crew dicts, each with a `'name'` and a `'job'` — e.g. `"Director"`).

In [ ]:
credits_data = {}   # movie_id -> credits dict
failed_credit_ids = []

for mid in df['id']:
    url = f"https://api.themoviedb.org/3/movie/{mid}/credits"
    params = {"api_key": api_key, "language": "en-US"}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        credits_data[mid] = response.json()
    else:
        print(f"Failed to fetch credits for movie_id {mid}: status {response.status_code}")
        failed_credit_ids.append(mid)

print(f"Successfully fetched credits for {len(credits_data)} out of {len(df)} movies.")


Extract the four fields we need from each movie's credits dictionary:
- `cast`: top 10 billed actor names, joined with `|`
- `cast_size`: total number of cast members listed
- `director`: name of the crew member whose `job` is `"Director"`
- `crew_size`: total number of crew members listed

In [ ]:
def get_cast_string(credits, top_n=10):
    cast_list = credits.get('cast', [])
    names = [person['name'] for person in cast_list[:top_n]]
    return "|".join(names) if names else None

def get_cast_size(credits):
    return len(credits.get('cast', []))

def get_director(credits):
    crew_list = credits.get('crew', [])
    for person in crew_list:
        if person.get('job') == 'Director':
            return person['name']
    return None

def get_crew_size(credits):
    return len(credits.get('crew', []))

df['cast'] = df['id'].map(lambda mid: get_cast_string(credits_data[mid]) if mid in credits_data else None)
df['cast_size'] = df['id'].map(lambda mid: get_cast_size(credits_data[mid]) if mid in credits_data else None)
df['director'] = df['id'].map(lambda mid: get_director(credits_data[mid]) if mid in credits_data else None)
df['crew_size'] = df['id'].map(lambda mid: get_crew_size(credits_data[mid]) if mid in credits_data else None)

df[['title', 'director', 'cast_size', 'crew_size']].head()


**Why `.map()` here, not a `for` loop or `.apply()` across the whole row?** `credits_data` is keyed by movie `id`, not by row position — `.map()` is the right tool for "look up a value for each entry in this column, using an external dictionary/function," which is exactly this case.

### Reorder columns & reset index

In [ ]:
final_columns = [
    'id', 'title', 'tagline', 'release_date', 'genres', 'belongs_to_collection',
    'original_language', 'budget_musd', 'revenue_musd', 'production_companies',
    'production_countries', 'vote_count', 'vote_average', 'popularity', 'runtime',
    'overview', 'spoken_languages', 'poster_path', 'cast', 'cast_size', 'director', 'crew_size'
]

df = df[final_columns]
df = df.reset_index(drop=True)

df.shape


### Save the cleaned dataset
Persist the cleaned DataFrame so Step 3 (KPI analysis) can load it directly, without re-fetching from the API every time.

In [ ]:
df.to_csv("../data/processed/tmdb_movies_clean.csv", index=False)
print("Saved to ../data/processed/tmdb_movies_clean.csv")
df.head()